# 01 – Data audit

Checks the Roboflow pork-rasher dataset before any training:

1. class balance per split,
2. photos that appear in more than one split (leakage),
3. a reproducible manual review of images with no defect labels.

Run in Google Colab. Add your Roboflow **private** API key to Colab Secrets as `ROBOFLOW_API_KEY` (key icon on the left) first.

In [ ]:
# Get the code (public repo, no token needed) and install the requirements.
# Safe to re-run: pulls the latest code if the repo is already cloned.
import os, sys

REPO = "/content/MMA3001-Project-repository"
if not os.path.isdir(REPO):
    !git clone https://github.com/rhal0027/MMA3001-Project-repository.git {REPO}
else:
    !git -C {REPO} pull
%cd {REPO}
!pip install -q -r requirements.txt

# Make the package importable straight away (an editable pip install
# only takes effect in Colab after a runtime restart).
if f"{REPO}/src" not in sys.path:
    sys.path.insert(0, f"{REPO}/src")


In [ ]:
# Download the dataset into data/raw/ (git-ignored)
from google.colab import userdata
from rasher_inspection import dataset as ds
from rasher_inspection import visualise as vis

root = ds.download_dataset(userdata.get("ROBOFLOW_API_KEY"), "data/raw/pork-rasher", version=4)
names = ds.load_class_names(root)
names

## 1. Class balance

In [ ]:
import pandas as pd

summary = ds.split_summary(root)
counts = pd.DataFrame({s: v["boxes"] for s, v in summary.items()})
info = pd.DataFrame({s: {k: v[k] for k in ("images", "sources", "empty_label", "missing_label")}
                     for s, v in summary.items()})
display(info)
display(counts)
counts.to_csv("results/class_counts.csv")
info.to_csv("results/split_info.csv")

## 2. Cross-split overlaps

Matched by Roboflow source id, so check each pair by eye: two different photos can share a file name.

In [ ]:
overlaps = ds.find_cross_split_overlaps(root)
print(len(overlaps), "source photos appear in more than one split")
vis.show_pairs(overlaps, class_names=names);

After looking at the pairs, list any that are **not** the same photo, then move the confirmed duplicates out of `train` (valid/test stay untouched). Run once with `dry_run=True` to check, then `False`.

In [ ]:
# Visual check (22 Oct 2026): all 13 file-name overlaps are DIFFERENT trays
# that happen to share a file name, so nothing is removed.
not_same_photo = ["1521_jpg", "2223_jpg", "2247_jpg", "3482_jpg", "6510_jpg",
                  "7277_jpg", "7506_jpg", "7508_jpg", "7510_jpg", "7555_jpg",
                  "7617_jpg", "8049_jpg", "9018_jpg"]
confirmed = [o for o in overlaps if o["source"] not in not_same_photo]

moved = ds.quarantine_overlaps(root, confirmed, "data/quarantine", from_split="train",
                               log_csv="results/leakage_removed.csv", dry_run=True)
print(len(confirmed), "confirmed duplicates;", len(moved), "train images would be moved")

## 2b. Content-based duplicate check

File names are not reliable here (the 13 matches above were all different trays), and a copy of the same tray under a *different* name would be missed. This compares the images themselves with a perceptual hash (dHash).

**First attempt (64-bit hash, threshold 6 bits):** 179 of 200 valid/test images were flagged, and the 20 closest pairs inspected were all *different* trays. The nearest-neighbour distances had no gap (median 3 bits): at 9×8 pixels the trays look almost identical. So the threshold must be **calibrated** rather than guessed.

**Calibration:** augmented copies in `train` share a source id, so they are *known* duplicates. Compare their distances with the distance from each image to its closest *different* tray. If the two distributions separate, the gap gives the threshold.

In [ ]:
import matplotlib.pyplot as plt

cal = {size: ds.hash_calibration(root, "train", size=size) for size in (8, 16)}   # ~1 min

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (size, c) in zip(axes, cal.items()):
    bins = range(0, max(c["same"] + c["different"]) + 2)
    ax.hist(c["different"], bins=bins, alpha=0.6, label="closest different tray")
    ax.hist(c["same"], bins=bins, alpha=0.6, label="augmented copy of same photo")
    ax.set_title(f"{size}x{size} dHash ({size*size} bits)")
    ax.set_xlabel("Hamming distance (bits)"); ax.set_ylabel("count"); ax.legend()
fig.tight_layout()
fig.savefig("results/dhash_calibration.png", dpi=150, bbox_inches="tight")
plt.show()

for size, c in cal.items():
    s, d = pd.Series(c["same"]), pd.Series(c["different"])
    print(f"{size}x{size}: same-photo max={s.max()}, 99th pct={s.quantile(.99):.0f} | "
          f"different-tray min={d.min()}, 1st pct={d.quantile(.01):.0f} | "
          f"different trays at or below same-photo max: {(d <= s.max()).sum()}")

Choose the hash size whose distributions separate, and set the threshold just above the largest same-photo distance. If neither size separates cleanly, report that: dHash is then not a reliable duplicate detector for this dataset, and the evidence against leakage rests on the file-name check plus manual inspection.

In [ ]:
SIZE = 16          # pick from the calibration plot
THRESHOLD = 20     # just above the same-photo maximum for that size

nearest = ds.find_near_duplicates(root, max_distance=SIZE * SIZE, size=SIZE)   # closest match for every valid/test image
matches = [m for m in nearest if m["distance"] <= THRESHOLD]
print(len(matches), "of", len(nearest), "valid/test images within", THRESHOLD, "bits of another image")
vis.show_matches(matches, class_names=names, max_rows=len(matches) or 1);

In [ ]:
# After looking at EVERY flagged pair: list the row numbers that ARE the same tray
same_tray_rows = []

pd.DataFrame([{"query": m["query"].name, "query_split": m["query_split"],
               "match": m["match"].name, "match_split": m["match_split"],
               "distance": m["distance"], "same_tray": i in same_tray_rows}
              for i, m in enumerate(matches)]).to_csv("results/near_duplicates.csv", index=False)
print(len(same_tray_rows), "confirmed duplicates of", len(matches), "flagged")

## 3. Manual review of unlabelled images

Criteria (written before reviewing):

- **clean** – no visible defect of any of the five classes
- **missed_defect** – a defect is visible but has no box (note which class)
- **unclear** – glare, blur or framing makes it impossible to judge

**Test split (22 Oct 2026):** all 12 unlabelled test images reviewed: 9 clean, 3 missed defects. Next: review a sample of the 684 unlabelled *train* images (set `SPLIT, SEED = "train", 1`).

In [ ]:
SPLIT, N, SEED = "test", 24, 0      # then: "train", 24, 1

unl = ds.find_unlabelled(root, SPLIT)
print({k: len(v) for k, v in unl.items()})
sample = ds.sample_for_review(unl["empty_label"], N, seed=SEED)
sheet = ds.write_review_sheet(sample, f"results/unlabelled_review_{SPLIT}_seed{SEED}.csv")
vis.show_grid(sample, cols=6, title=f"Unlabelled {SPLIT} images (seed {SEED})");

In [ ]:
from rasher_inspection.stats import wilson_interval

# Verdicts keyed by the photo number at the start of the file name.
# Anything not listed is recorded as "clean".
verdicts = {
    "test": {
        "4088": ("missed_defect", "packaging-error"),
        "7471": ("missed_defect", "unsealed"),
        "7325": ("missed_defect", "twisted-meat"),
    },
    "train": {
        # "1234": ("missed_defect", "loose-meat"),
        # "5678": ("unclear", "glare"),
    },
}

review = pd.read_csv(sheet)
number = review["image"].str.split("_").str[0]
v = number.map(lambda n: verdicts[SPLIT].get(n, ("clean", "")))
review["verdict"] = [x[0] for x in v]
review["notes"] = [x[1] for x in v]
review.to_csv(sheet, index=False)

missed = (review["verdict"] == "missed_defect").sum()
lo, hi = wilson_interval(missed, len(review))
print(review["verdict"].value_counts())
print(f"Missed-defect rate: {missed}/{len(review)} = {missed/len(review):.0%} "
      f"(95% CI {lo:.0%}-{hi:.0%})")
# Download results/*.csv and results/*.png from the Colab file browser and commit them.